In [0]:
--use catalog lakehouse_dev;
--use schema datasphere_test;
create or refresh streaming table orders_bronze
as 
select *, current_timestamp() as processing_time,
_metadata.file_name as source_file
from stream read_files("/Volumes/lakehouse_dev/datasphere_test/managedvolume/orders", format=>"json", schema => "item string, order_id string, quantity bigint, price double, txdate timestamp");

In [0]:
use catalog lakehouse_dev;
use schema datasphere_test;
create or refresh streaming table orders_silver
(constraint has_quantity expect (quantity > 1) on violation drop row )
comment "Append only orders"
tblproperties("quality" = 'silver')
as 
select 
* from stream(LIVE.orders_bronze)


In [0]:
use catalog lakehouse_dev;
use schema datasphere_test;
create or replace materialized view orders_gold
as select 
count(*) as total_orders
from LIVE.orders_silver